In [0]:
%pip install gradio plotly

In [0]:
dbutils.library.restartPython()

In [0]:
import gradio as gr
import pandas as pd
from pyspark.sql.functions import col, countDistinct, sum as spark_sum, when, expr
import plotly.express as px
import plotly.graph_objects as go

In [0]:
# Load transaction data once
df = spark.read.csv("/Volumes/workspace/default/loan/transaction_data.csv", header=True, inferSchema=True)
print(f"Data loaded: {df.count()} transactions")

In [0]:
def analyze_risk(flagged_users_input, connectivity_weight, velocity_weight, first_time_weight, top_n):
    """
    Analyze receiver risk based on flagged users and risk parameters
    """
    try:
        # Parse flagged UserIds from comma-separated string
        if not flagged_users_input or flagged_users_input.strip() == "":
            return "<h3 style='color: red;'>⚠️ Please enter at least one flagged UserID</h3>", None
        
        flagged_nodes = [int(x.strip()) for x in flagged_users_input.split(",") if x.strip()]
        
        if len(flagged_nodes) == 0:
            return "<h3 style='color: red;'>⚠️ No valid UserIDs provided</h3>", None
        
        # Count how many flagged nodes each receiver (ItemCode) is connected to
        connectivity_df = df.filter(col("UserId").isin(flagged_nodes)) \
            .groupBy("ItemCode") \
            .agg(countDistinct("UserId").alias("flagged_connections"))
        
        # Calculate velocity of incoming funds
        velocity_df = df.groupBy("ItemCode") \
            .agg(
                spark_sum(col("NumberOfItemsPurchased") * col("CostPerItem")).alias("total_incoming_funds"),
                countDistinct("TransactionId").alias("transaction_count")
            )
        
        # Identify first-time interactions
        first_time_df = df.groupBy("ItemCode") \
            .agg(countDistinct("TransactionId").alias("transaction_count")) \
            .filter(col("transaction_count") == 1) \
            .select("ItemCode")
        
        first_time_items = [row.ItemCode for row in first_time_df.collect()]
        
        # Combine risk factors
        risk_df = connectivity_df.join(velocity_df, "ItemCode", "left") \
            .withColumn("first_time_interaction", when(col("ItemCode").isin(first_time_items), 1).otherwise(0)) \
            .withColumn("risk_score", 
                col("flagged_connections") * connectivity_weight +
                (col("total_incoming_funds") / 1000) * velocity_weight +
                col("first_time_interaction") * first_time_weight
            ) \
            .withColumn("reason",
                when(col("flagged_connections") > 0, "Linked to high-risk nodes")
                .when(col("first_time_interaction") == 1, "First-time interaction")
                .otherwise("Normal")
            )
        
        # Get top N risky items
        top_risky = risk_df.orderBy(col("risk_score").desc()).limit(top_n).toPandas()
        
        # Format results
        top_risky["total_incoming_funds"] = top_risky["total_incoming_funds"].round(2)
        top_risky["risk_score"] = top_risky["risk_score"].round(2)
        
        # Create summary HTML
        summary_html = f"""
        <div style='padding: 20px; background-color: #f0f8ff; border-radius: 10px; border-left: 5px solid #4169e1;'>
            <h2 style='color: #4169e1; margin-top: 0;'>🔍 Risk Analysis Summary</h2>
            <p style='font-size: 16px;'><b>Flagged UserIDs:</b> {', '.join(map(str, flagged_nodes))}</p>
            <p style='font-size: 16px;'><b>Total Risky Items Found:</b> {len(top_risky)}</p>
            <p style='font-size: 16px;'><b>Highest Risk Score:</b> {top_risky['risk_score'].max():.2f}</p>
            <p style='font-size: 16px;'><b>Risk Parameters:</b> Connectivity={connectivity_weight}x, Velocity={velocity_weight}x, First-time={first_time_weight}x</p>
        </div>
        """
        
        # Create visualization
        fig = go.Figure()
        
        # Add bar chart for risk scores
        fig.add_trace(go.Bar(
            x=top_risky['ItemCode'].astype(str),
            y=top_risky['risk_score'],
            marker_color=top_risky['risk_score'],
            marker_colorscale='Reds',
            text=top_risky['risk_score'].round(2),
            textposition='outside',
            name='Risk Score'
        ))
        
        fig.update_layout(
            title=f'Top {top_n} Highest Risk Items',
            xaxis_title='Item Code',
            yaxis_title='Risk Score',
            height=400,
            showlegend=False
        )
        
        return summary_html, fig, top_risky
    
    except Exception as e:
        return f"<h3 style='color: red;'>⚠️ Error: {str(e)}</h3>", None, None

print("Risk analysis function defined")

In [0]:
# Create Gradio Dashboard
with gr.Blocks(title="Receiver Risk Analysis Dashboard", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🛡️ Receiver Risk Analysis Dashboard
        ### Monitor and analyze transaction risk based on flagged user connections
        """
    )
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📊 Analysis Parameters")
            
            flagged_users = gr.Textbox(
                label="Flagged UserIDs (comma-separated)",
                placeholder="e.g., 267099, 350805, 372750",
                value="267099, 350805, 372750",
                info="Enter UserIDs of flagged/suspicious users"
            )
            
            gr.Markdown("#### Risk Weight Factors")
            
            connectivity_weight = gr.Slider(
                minimum=0,
                maximum=10,
                value=2,
                step=0.5,
                label="Connectivity Weight",
                info="Weight for connections to flagged users"
            )
            
            velocity_weight = gr.Slider(
                minimum=0,
                maximum=5,
                value=1,
                step=0.1,
                label="Velocity Weight",
                info="Weight for total incoming funds"
            )
            
            first_time_weight = gr.Slider(
                minimum=0,
                maximum=10,
                value=3,
                step=0.5,
                label="First-time Interaction Weight",
                info="Weight for first-time receivers"
            )
            
            top_n = gr.Slider(
                minimum=5,
                maximum=100,
                value=20,
                step=5,
                label="Number of Top Risky Items to Show",
                info="How many results to display"
            )
            
            analyze_btn = gr.Button("🔍 Analyze Risk", variant="primary", size="lg")
        
        with gr.Column(scale=2):
            gr.Markdown("### 📈 Results")
            summary_output = gr.HTML(label="Summary")
            chart_output = gr.Plot(label="Risk Distribution")
    
    with gr.Row():
        table_output = gr.Dataframe(
            label="Detailed Risk Breakdown",
            headers=["ItemCode", "Risk Score", "Reason", "Flagged Connections", "Total Incoming Funds", "Transaction Count", "First-time Interaction"],
            interactive=False
        )
    
    analyze_btn.click(
        fn=analyze_risk,
        inputs=[flagged_users, connectivity_weight, velocity_weight, first_time_weight, top_n],
        outputs=[summary_output, chart_output, table_output]
    )
    
    gr.Markdown(
        """
        ---
        **How to use:**
        1. Enter UserIDs of flagged/suspicious users (comma-separated)
        2. Adjust risk weight parameters based on your risk model
        3. Click "Analyze Risk" to see results
        4. Review the top risky items and their connection patterns
        
        **Risk Score Formula:** `(Flagged Connections × Weight) + (Total Funds / 1000 × Weight) + (First-time × Weight)`
        """
    )

# Launch with public sharing enabled
demo.launch(share=True, inline=True)